In [2]:
## Compare results of classifying images between Tensorflow and PyTorch
# Tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models
import time

In [3]:
# Data Preprocessing
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

170498071/170498071 [==============================] - 17s 0us/step


In [6]:
# Configure CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [8]:
# Train model
start_time = time.time()
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.1
)

tf_train_time = time.time() - start_time
print(f"Training time of TensorFlow: {tf_train_time:.2f} seconds")

Epoch 1/5
704/704 [==============================] - 14s 19ms/step - loss: 1.4513 - accuracy: 0.4810 - val_loss: 1.1843 - val_accuracy: 0.5888
Epoch 2/5
704/704 [==============================] - 12s 18ms/step - loss: 1.0807 - accuracy: 0.6211 - val_loss: 1.0167 - val_accuracy: 0.6428
Epoch 3/5
704/704 [==============================] - 12s 18ms/step - loss: 0.9357 - accuracy: 0.6735 - val_loss: 0.9545 - val_accuracy: 0.6732
Epoch 4/5
704/704 [==============================] - 12s 17ms/step - loss: 0.8392 - accuracy: 0.7057 - val_loss: 0.8892 - val_accuracy: 0.6968
Epoch 5/5
704/704 [==============================] - 12s 17ms/step - loss: 0.7543 - accuracy: 0.7366 - val_loss: 0.8857 - val_accuracy: 0.6982
Training time of TensorFlow: 66.58 seconds


In [10]:
# Test accuracy
test_loss, tf_test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy of Tensorflow:, {tf_test_accuracy:.4f}')

Test accuracy of Tensorflow:, 0.6886


In [11]:
# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

In [12]:
# Load data
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 170498071/170498071 [47:37<00:00, 59664.65it/s] 


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [13]:
# Class CNN
class CNNmodel(nn.Module):
    def __init__(self):
        super(CNNmodel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.softmax(self.fc2(x))
        return x
    
model = CNNmodel()

In [14]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
# Train model
start_time = time.time()
model.train()
for epoch in range(5):
    running_loss = 0.0
    current = 0
    total = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs, labels

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        current += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = current / total
    print(f"Epoch {epoch+1} loss: {epoch_loss:.4f}, acc: {epoch_acc:.4f}")

torch_train_time = time.time() - start_time
print(f"Training time of PyTorch: {torch_train_time:.4f} seconds")

Epoch 1 loss: 2.1064, acc: 0.3478
Epoch 2 loss: 1.9697, acc: 0.4903
Epoch 3 loss: 1.9146, acc: 0.5467
Epoch 4 loss: 1.8796, acc: 0.5819
Epoch 5 loss: 1.8533, acc: 0.6081
Training time of PyTorch: 110.1189 seconds


In [16]:
# test
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs, labels
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

torch_test_acc = correct / total
print(f"PyTorch测试准确率：{torch_test_acc:.4f}")

PyTorch测试准确率：0.5982
